In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

###cust_info###

***fill null cust_id***

In [0]:
##drop duplicates cst_id based on fname, lname
df_cust_info = spark.read.table("medallionsales.bronze.cust_info")

w = Window.partitionBy("cst_id").orderBy(
    F.col("cst_firstname").isNull().cast("int"),
    F.col("cst_lastname").isNull().cast("int")
)

df_cust_info = (
    df_cust_info
    .withColumn("rn", F.row_number().over(w))\
    .filter(F.col("rn") == 1)\
    .drop("rn")
)

In [0]:
df_cust_info.show()
#lower and trim all string columns except cst_key
for field in df_cust_info.schema.fields:
    if field.dataType == StringType() and field.name != "cst_key":
        df_cust_info = df_cust_info.withColumn(field.name, F.trim(F.lower(F.col(field.name))))

+------+----------+-------------+------------+------------------+--------+---------------+--------------------+--------------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date|   _ingest_timestamp|        _source_file|
+------+----------+-------------+------------+------------------+--------+---------------+--------------------+--------------------+
|  NULL|     SF566|         NULL|        NULL|              NULL|    NULL|           NULL|2026-09-28 05:09:...|abfss://sink-sys@...|
| 11000|AW00011000|          Jon|       Yang |                 M|       M|     2025-10-06|2026-09-28 05:09:...|abfss://sink-sys@...|
| 11001|AW00011001|       Eugene|     Huang  |                 S|       M|     2025-10-06|2026-09-28 05:09:...|abfss://sink-sys@...|
| 11002|AW00011002|        Ruben|      Torres|                 M|       M|     2025-10-06|2026-09-28 05:09:...|abfss://sink-sys@...|
| 11003|AW00011003|      Christy|         Zhu|                 S|    

***handling enrichment cust_az12***


In [0]:
df_cust_az12 = spark.read.table("medallionsales.bronze.cust_az12")

##normalize gender value
df_cust_az12 = df_cust_az12.withColumn(
    "GEN",
    F.when(
        F.trim(F.col("GEN")) == "",
        F.lit(None)
    )
    .when(
        F.trim(F.col("GEN")) == "Female", "F"
    )
    .when(
        F.trim(F.col("GEN")) == "Male", "M"
    )
    .otherwise(F.trim(F.col("GEN")))
)


In [0]:
df_cust_az12.groupBy("GEN").count().show()

+----+-----+
| GEN|count|
+----+-----+
|   M| 8612|
|   F| 8396|
|NULL| 1476|
+----+-----+



In [0]:
##noramlize CID values
df_cust_az12 = df_cust_az12.withColumn(
    "CID",
    F.regexp_replace(F.col("CID"), r"^NAS", "")
)

In [0]:
df_cust_az12.show()

+----------+----------+---+--------------------+--------------------+
|       CID|     BDATE|GEN|   _ingest_timestamp|        _source_file|
+----------+----------+---+--------------------+--------------------+
|AW00011000|1971-10-06|  M|2026-09-28 05:09:...|abfss://sink-sys@...|
|AW00011001|1976-05-10|  M|2026-09-28 05:09:...|abfss://sink-sys@...|
|AW00011002|1971-02-09|  M|2026-09-28 05:09:...|abfss://sink-sys@...|
|AW00011003|1973-08-14|  F|2026-09-28 05:09:...|abfss://sink-sys@...|
|AW00011004|1979-08-05|  F|2026-09-28 05:09:...|abfss://sink-sys@...|
|AW00011005|1976-08-01|  M|2026-09-28 05:09:...|abfss://sink-sys@...|
|AW00011006|1976-12-02|  F|2026-09-28 05:09:...|abfss://sink-sys@...|
|AW00011007|1969-11-06|  M|2026-09-28 05:09:...|abfss://sink-sys@...|
|AW00011008|1975-07-04|  F|2026-09-28 05:09:...|abfss://sink-sys@...|
|AW00011009|1969-09-29|  M|2026-09-28 05:09:...|abfss://sink-sys@...|
|AW00011010|1969-08-05|  F|2026-09-28 05:09:...|abfss://sink-sys@...|
|AW00011011|1969-05-

***fill gender nulls in customer information***

In [0]:
##fill null values in customer information gender field using enrichment df_cust_az12
df_cust_info = (
    df_cust_info.alias("ci")\
        .join(
            df_cust_az12.select("CID", "GEN").alias("ce"),
            how="left",
            on = F.col("ci.cst_key") == F.col("ce.CID")
        )
        .withColumn(
            "cst_gndr",
            F.coalesce(F.col("ci.cst_gndr"), F.col("ce.GEN"))
        )
        .drop("GEN", "CID")\
)

In [0]:
df_cust_info.filter(F.col("cst_gndr").isNull()).count()


16

In [0]:
##fill back null values in enrichment df_cust_az12 gender field using customer information
df_cust_az12 = (
    df_cust_az12.alias("ce")\
        .join(
            df_cust_info.select("cst_key", "cst_gndr").alias("ci"),
            how="left",
            on = F.col("ci.cst_key") == F.col("ce.CID")
        )
        .withColumn(
            "GEN",
            F.coalesce(F.col("ce.GEN"), F.col("ci.cst_gndr"))
        )
        .drop("cst_key", "cst_gndr")\
)
df_cust_az12.filter(F.col("GEN").isNull()).count()

15

***handling loca_a101***

In [0]:
df_loc_a101 = (
    spark.read.table("medallionsales.bronze.loc_a101")
    .fillna("Unknown", subset=["CNTRY"])
    .withColumn(
        "CID",
        F.regexp_replace("CID", "-", "")
    )
    .withColumn(
        "CNTRY",
        F.when(F.trim("CNTRY") == "", None)
         .when(F.trim("CNTRY").isin("USA", "US"), "United States")
         .when(F.trim("CNTRY") == "DE", "Germany")
         .otherwise(F.trim("CNTRY"))
    )
)

In [0]:
df_loc_a101.select("CNTRY").distinct().show()

+--------------+
|         CNTRY|
+--------------+
|     Australia|
| United States|
|        Canada|
|       Germany|
|United Kingdom|
|        France|
|       Unknown|
|          NULL|
+--------------+



In [0]:
##create final data frame for customer with all data the available data
RENAME_MAP = {
    "cst_id"  : "customer_id",
    "cst_key" : "customer_key",
    "cst_firstname" : "first_name",
    "cst_lastname" : "last_name",
    "cst_marital_status" : "martial_status",
    "cst_gndr" : "gender",
    "cst_create_date" : "create_date",
    "cst_cntry" : "country"
 }

silver_customer = (
    df_cust_info.alias("ci")\
        .join(
            df_loc_a101.select("CID", "CNTRY").alias("lc"),
            how = "left",
            on = F.col("ci.cst_key") == F.col("lc.CID")
        )
        .withColumns({
            "cst_cntry" :F.lower(F.col("lc.CNTRY")),
            "cst_firstname": F.trim(F.lower(F.col("cst_firstname"))),
            "cst_lastname": F.trim(F.lower(F.col("cst_lastname")))
        })
        .drop("CID", "CNTRY")
)
silver_customer = silver_customer.select(
    *[c for c in df_cust_info.columns if c not in ["cst_cntry", "_ingest_timestamp", "_source_file"]],
    "cst_cntry",
    "_ingest_timestamp",
    "_source_file"
)  

for old, new in RENAME_MAP.items():
    silver_customer = silver_customer.withColumnRenamed(old, new)      
    
silver_customer.show()

+-----------+------------+----------+---------+--------------+------+-----------+-------------+--------------------+--------------------+
|customer_id|customer_key|first_name|last_name|martial_status|gender|create_date|      country|   _ingest_timestamp|        _source_file|
+-----------+------------+----------+---------+--------------+------+-----------+-------------+--------------------+--------------------+
|       NULL|       SF566|      NULL|     NULL|          NULL|  NULL|       NULL|         NULL|2026-09-28 05:09:...|abfss://sink-sys@...|
|      11000|  AW00011000|       jon|     yang|             m|     m| 2025-10-06|    australia|2026-09-28 05:09:...|abfss://sink-sys@...|
|      11001|  AW00011001|    eugene|    huang|             s|     m| 2025-10-06|    australia|2026-09-28 05:09:...|abfss://sink-sys@...|
|      11002|  AW00011002|     ruben|   torres|             m|     m| 2025-10-06|    australia|2026-09-28 05:09:...|abfss://sink-sys@...|
|      11003|  AW00011003|   chris

In [0]:
silver_customer.dtypes

[('customer_id', 'int'),
 ('customer_key', 'string'),
 ('first_name', 'string'),
 ('last_name', 'string'),
 ('martial_status', 'string'),
 ('gender', 'string'),
 ('create_date', 'date'),
 ('country', 'string'),
 ('_ingest_timestamp', 'timestamp'),
 ('_source_file', 'string')]

###Product_inforamtion###

***handling px_cat_g1v2***

In [0]:
df_px_cat_g1v2 = spark.read.table("medallionsales.bronze.px_cat_g1v2")

## Trim and lowercase all string columns
for field in df_px_cat_g1v2.schema.fields:
    if field.dataType == StringType() and field.name != "ID":
        df_px_cat_g1v2 = df_px_cat_g1v2.withColumn(field.name, F.trim(F.lower(F.col(field.name))))

df_px_cat_g1v2.show(5)


+-----+-----------+-----------------+-----------+--------------------+--------------------+
|   ID|        CAT|           SUBCAT|MAINTENANCE|   _ingest_timestamp|        _source_file|
+-----+-----------+-----------------+-----------+--------------------+--------------------+
|AC_BR|accessories|       bike racks|        yes|2026-09-28 05:09:...|abfss://sink-sys@...|
|AC_BS|accessories|      bike stands|         no|2026-09-28 05:09:...|abfss://sink-sys@...|
|AC_BC|accessories|bottles and cages|         no|2026-09-28 05:09:...|abfss://sink-sys@...|
|AC_CL|accessories|         cleaners|        yes|2026-09-28 05:09:...|abfss://sink-sys@...|
|AC_FE|accessories|          fenders|         no|2026-09-28 05:09:...|abfss://sink-sys@...|
+-----+-----------+-----------------+-----------+--------------------+--------------------+
only showing top 5 rows


***handling product info***

In [0]:
df_prd_info = spark.read.table("medallionsales.bronze.prd_info")

## Trim and lowercase all string columns
for field in df_prd_info.schema.fields:
    if field.dataType == StringType() and field.name != "prd_key":
        df_prd_info = df_prd_info.withColumn(field.name, F.trim(F.lower(F.col(field.name))))

In [0]:
# w = Window.partitionBy("prd_key").orderBy("prd_start_dt")

# df_prd_info = (
#     df_prd_info

#     # 1. Validate source date range
#     .withColumn(
#         "is_valid_date_range",
#         ~(
#             (F.col("prd_end_dt").isNotNull()) &
#             (F.col("prd_start_dt") > F.col("prd_end_dt"))
#         )
#     )

#     # 2. Get next SCD version
#     .withColumn(
#         "next_start_dt",
#         F.lead("prd_start_dt").over(w)
#     )

#     # 3. Derive corrected end date only when source is invalid
#     .withColumn(
#         "effective_end_date",
#         F.when(
#             ~F.col("is_valid_date_range"),
#             F.col("next_start_dt")
#         ).otherwise(
#             F.col("prd_end_dt")
#         )
#     )


In [0]:
df_prd_info = (
    df_prd_info
    .withColumn(
        "ID",
        F.concat_ws(
            "_",
            F.split("prd_key", "-").getItem(0),
            F.split("prd_key", "-").getItem(1)
        )
    )
    .withColumn(
        "product_business_key",
        F.concat_ws(
            "-",
            F.slice(F.split("prd_key", "-"), 3, 100)
        )
    )
)

In [0]:
#join operation
df_prd_info = (
    df_prd_info.alias("pi")\
        .join(
            df_px_cat_g1v2.select("ID","CAT", "SUBCAT", "MAINTENANCE").alias("pc"),
            how = "left",
            on = F.col("pi.ID") == F.col("pc.ID")
        )
        .select(
        "pi.prd_id",
        "pi.prd_key",
        "pi.product_business_key",
        "pi.prd_nm",
        "pc.CAT",
        "pc.SUBCAT",
        "pc.MAINTENANCE",
        "pi.prd_cost",
        "pi.prd_line",
        "pi.prd_start_dt",
        "pi.prd_end_dt",
        "pi._ingest_timestamp"
    )
)

In [0]:
##Rename columns
RENAME_MAP = {
    "prd_id" : "product_id",
    "prd_key" : "product_key",
    "prd_nm" : "product_name",
    "CAT" : "category",
    "SUBCAT" : "sub_category",
    "MAINTENANCE" : "maintenance",
    "prd_cost" : "product_cost",
    "prd_line" : "product_line",
    "prd_start_dt" : "start_date",
    "prd_end_dt" : "end_date",
    "_ingest_timestamp" : "ingest_timestamp"
}

##SCD Type 2 
#####add column is_valid_date_range to detect if the row has valid or invalid date range
##### detect that the values were swapped in invalid rows 
#####add column original_start_dt to store the original start date before modification
#####swap star date and end date in invalid rows


df_prd_info = (
    df_prd_info
    .withColumn(
        "is_valid_date_range",
        ~(
            (F.col("prd_end_dt").isNotNull()) &
            (F.col("prd_start_dt") > F.col("prd_end_dt"))
        )
    )
    .withColumn(
        "original_start_dt",
        F.col("prd_start_dt")
    )
    .withColumn(
        "prd_start_dt",
        F.when(
            ~F.col("is_valid_date_range"),
            F.col("prd_end_dt")
        ).otherwise(F.col("prd_start_dt"))
    )
    .withColumn(
        "prd_end_dt",
        F.when(
            ~F.col("is_valid_date_range"),
            F.col("original_start_dt")
        ).otherwise(F.col("prd_end_dt"))
    )
    .withColumns({
    "prd_start_dt": F.date_format(
        F.col("prd_start_dt"),
        "yyyyMMdd"
    ).cast("int"),

    "prd_end_dt": F.date_format(
        F.col("prd_end_dt"),
        "yyyyMMdd"
    ).cast("int")
})
    .drop("original_start_dt")
)

##Renaming operation
for old, name in RENAME_MAP.items():
    df_prd_info = df_prd_info.withColumnRenamed(old, name)




In [0]:
silver_products = df_prd_info

###Sales###

In [0]:
df_sales = spark.read.table("medallionsales.bronze.sales_details")


In [0]:
##outlier
df_sales.filter(
    (F.col("sls_order_dt") != 0) &
    (F.col("sls_ship_dt") != 0)
).filter(
    ~F.col("sls_order_dt").cast("string").rlike(r"^\d{8}$") |
    ~F.col("sls_ship_dt").cast("string").rlike(r"^\d{8}$")
).select(
    "sls_order_dt",
    "sls_ship_dt"
).show(50, False)

+------------+-----------+
|sls_order_dt|sls_ship_dt|
+------------+-----------+
|32154       |20131102   |
|5489        |20131102   |
+------------+-----------+



In [0]:
##any value not in format yyyyMMdd got 0 (this include the outliers)
df_sales = df_sales.withColumn(
    "sls_order_dt",
    F.when(
        ~F.col("sls_order_dt").cast("string").rlike(r"^\d{8}$"),
        F.lit(0)
    ).otherwise(F.col("sls_order_dt"))
)

###with helping shipdate derived orderdate for all rows with 0 orderdate (as the business takes 7 days to ship the product)
df_sales = (
    df_sales
    .withColumn(
        "order_date",
        F.when(
            F.col("sls_order_dt") == 0,
            F.date_sub(
                F.to_date(
                    F.col("sls_ship_dt").cast("string"),
                    "yyyyMMdd"
                ),
                7
            )
        )
    )
    .withColumn(
        "sls_order_dt",
        F.when(
            F.col("sls_order_dt") == 0,
            F.date_format(
                F.col("order_date"),
                "yyyyMMdd"
            ).cast("int")
        ).otherwise(F.col("sls_order_dt"))
    )
    .drop("order_date")
)

In [0]:
##no invalid orderdate
df_sales.filter(F.col("sls_order_dt") == 0).show()

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+-----------------+------------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|_ingest_timestamp|_source_file|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+-----------------+------------+
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+-----------------+------------+



In [0]:
##handling prices
silver_sales = (
    df_sales
    .withColumn(
        "sls_price",
        F.when(
            (F.col("sls_price").isNull()) | (F.col("sls_price") <= 0),
            F.when(
                F.col("sls_quantity") != 0,
                F.col("sls_sales") / F.col("sls_quantity")
            ).otherwise(None)
        ).otherwise(F.col("sls_price"))
    )
)


In [0]:
##Renaming columns
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    silver_sales = silver_sales.withColumnRenamed(old_name, new_name)
     

In [0]:
silver_sales.show()

+------------+--------------+-----------+----------+---------+--------+------------+--------+------+--------------------+--------------------+
|order_number|product_number|customer_id|order_date|ship_date|due_date|sales_amount|quantity| price|   _ingest_timestamp|        _source_file|
+------------+--------------+-----------+----------+---------+--------+------------+--------+------+--------------------+--------------------+
|     SO43697|    BK-R93R-62|      21768|  20101229| 20110105|20110110|        3578|       1|3578.0|2026-09-28 05:09:...|abfss://sink-sys@...|
|     SO43698|    BK-M82S-44|      28389|  20101229| 20110105|20110110|        3400|       1|3400.0|2026-09-28 05:09:...|abfss://sink-sys@...|
|     SO43699|    BK-M82S-44|      25863|  20101229| 20110105|20110110|        3400|       1|3400.0|2026-09-28 05:09:...|abfss://sink-sys@...|
|     SO43700|    BK-R50B-62|      14501|  20101229| 20110105|20110110|         699|       1| 699.0|2026-09-28 05:09:...|abfss://sink-sys@...|

###Date###

######save fact and dimensions######

In [0]:
dim_fact_tables = {
    "silver_sales": silver_sales,
    "silver_customer": silver_customer,
    "silver_product": silver_products
}

for table_name, df in dim_fact_tables.items():
    (
        df.write
        .mode("overwrite")
        .format("delta")
        .option("mergeSchema", "true")
        .saveAsTable(f"medallionsales.silver.{table_name}")
    )